In [1]:
# Cell 1 — Setup (same logic as your KAN-AD notebook)

!pip install --upgrade pip
!pip install toml torch torchinfo optuna
!pip install git+https://github.com/CSTCloudOps/EasyTSAD.git

# Clone repos
!rm -rf KAN-AD
!rm -rf datasets

!git clone https://github.com/CSTCloudOps/KAN-AD.git
!git clone https://github.com/CSTCloudOps/datasets.git
!mv datasets KAN-AD/datasets

%cd KAN-AD

# Path setup
import sys, os

project_path = os.getcwd()
if project_path not in sys.path:
    sys.path.append(project_path)

print("🟢 Paths configured.")
print("Current working directory:", project_path)

# Fix EasyTSAD import bug
!sed -i 's/TSData、/TSData/g; s/TSData,*/TSData/g' \
    /usr/local/lib/python3.12/dist-packages/EasyTSAD/DataFactory/__init__.py

# Test imports
from EasyTSAD.Controller import TSADController
from EasyTSAD.DataFactory import TSData
from EasyTSAD.Methods import BaseMethod

print("✅ EasyTSAD ready")

  Cloning https://github.com/CSTCloudOps/EasyTSAD.git to /tmp/pip-req-build-9szoqju4
  Running command git clone --filter=blob:none --quiet https://github.com/CSTCloudOps/EasyTSAD.git /tmp/pip-req-build-9szoqju4
  Resolved https://github.com/CSTCloudOps/EasyTSAD.git to commit 507e5533a142eec7eece4372c55e83bb3faff8a1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Cloning into 'KAN-AD'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 61.77 KiB | 930.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
Cloning into 'datasets'...
remote: Enumerating objects: 4503, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 4503 (delta 2), reused 0 (delta 0), pack-r

In [2]:
# Cell 2 — DLinear backbone (forecasting → anomaly detection)

import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Moving Average (trend extractor)
# -----------------------------
class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2

    def forward(self, x):
        # x: (B, W)
        x = x.unsqueeze(1)  # (B, 1, W)

        # replicate padding (important for stability)
        x = F.pad(x, (self.padding, self.padding), mode='replicate')

        # avg pooling
        x = F.avg_pool1d(x, kernel_size=self.kernel_size, stride=1)

        return x.squeeze(1)  # (B, W)


# -----------------------------
# Series Decomposition
# -----------------------------
class SeriesDecomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)

    def forward(self, x):
        # x: (B, W)
        trend = self.moving_avg(x)
        seasonal = x - trend
        return seasonal, trend


# -----------------------------
# DLinear Model
# -----------------------------
class DLinearModel(nn.Module):
    """
    DLinear adapted for anomaly detection:
    input: window (W)
    output: next value (1)
    """

    def __init__(self, window, kernel_size=25):
        super().__init__()

        self.window = int(window)

        # decomposition
        self.decomp = SeriesDecomp(kernel_size)

        # linear heads
        self.linear_seasonal = nn.Linear(window, 1)
        self.linear_trend = nn.Linear(window, 1)

    def forward(self, x):
        """
        x: (B*F, W)
        """

        seasonal, trend = self.decomp(x)

        out_seasonal = self.linear_seasonal(seasonal)
        out_trend = self.linear_trend(trend)

        out = out_seasonal + out_trend  # final prediction

        return out  # (B*F, 1)


print("✅ DLinear model ready")

✅ DLinear model ready


In [3]:
# Cell 3 — DLinear as EasyTSAD Method

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import tqdm

from torch.utils.data import DataLoader
from EasyTSAD.Methods import BaseMethod

# -----------------------------
# DLinear EasyTSAD Wrapper
# -----------------------------
class DLinear_TSAD(BaseMethod):

    def __init__(self, params: dict) -> None:
        super().__init__()

        self.__anomaly_score = None

        self.window = int(params["window"])
        self.batch_size = int(params["batch_size"])
        self.epochs = int(params["epochs"])
        self.lr = float(params["lr"])
        self.patience = int(params.get("patience", 5))

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # ✅ DLinear backbone
        self.model = DLinearModel(window=self.window).to(self.device)

        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=5, gamma=0.75)
        self.mse_loss = nn.MSELoss()

        self.best_state = None

    # -----------------------------
    # Forward batch
    # -----------------------------
    def _forward_batch(self, x, target):
        B, W, F = x.shape

        # reshape to channel-independent
        x_1d = x.permute(0, 2, 1).reshape(B * F, W)
        t_1d = target.reshape(B * F, 1)

        out = self.model(x_1d)

        loss = self.mse_loss(out, t_1d)

        # anomaly score = prediction error
        pred_err = (out - t_1d).abs().reshape(B, F).max(dim=1).values

        return pred_err, loss

    # -----------------------------
    # Train + Valid
    # -----------------------------
    def train_valid_phase(self, tsTrain):

        train_loader = DataLoader(
            MTSWindowDataset(tsTrain, "train", self.window),
            batch_size=self.batch_size,
            shuffle=True
        )

        valid_loader = DataLoader(
            MTSWindowDataset(tsTrain, "valid", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        best_valid = float("inf")
        patience_counter = 0

        for epoch in range(1, self.epochs + 1):

            self.model.train()
            train_losses = []

            for x, y in tqdm.tqdm(train_loader, desc=f"Train {epoch}"):

                x = x.to(self.device)
                y = y.to(self.device)

                self.optimizer.zero_grad(set_to_none=True)

                _, loss = self._forward_batch(x, y)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=5.0)

                self.optimizer.step()
                train_losses.append(float(loss.item()))

            # VALID
            self.model.eval()
            valid_losses = []

            with torch.no_grad():
                for x, y in tqdm.tqdm(valid_loader, desc=f"Valid {epoch}"):

                    x = x.to(self.device)
                    y = y.to(self.device)

                    _, loss = self._forward_batch(x, y)
                    valid_losses.append(float(loss.item()))

            train_loss = np.mean(train_losses)
            valid_loss = np.mean(valid_losses)

            print(f"Epoch {epoch} | train={train_loss:.6f} | valid={valid_loss:.6f}")

            self.scheduler.step()

            # Early stopping
            if valid_loss < best_valid:
                best_valid = valid_loss
                patience_counter = 0

                self.best_state = {
                    "model": {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
                }
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print("Early stopping")
                    break

        # restore best model
        assert self.best_state is not None
        self.model.load_state_dict(self.best_state["model"])

    # -----------------------------
    # Test phase
    # -----------------------------
    def test_phase(self, tsData):

        test_loader = DataLoader(
            MTSWindowDataset(tsData, "test", self.window),
            batch_size=self.batch_size,
            shuffle=False
        )

        self.model.eval()

        scores = []

        with torch.no_grad():
            for x, y in test_loader:

                x = x.to(self.device)
                y = y.to(self.device)

                pred_err, _ = self._forward_batch(x, y)
                scores.append(pred_err.cpu().numpy())

        scores = np.concatenate(scores) if scores else np.array([])

        # pad to match original length
        if len(scores) == 0:
            padded = np.zeros(len(tsData.test))
        else:
            prefix = np.full(self.window, scores[0])
            padded = np.concatenate([prefix, scores])
            padded = padded[:len(tsData.test)]

        self.__anomaly_score = padded.astype(np.float64)

    # -----------------------------
    def anomaly_score(self) -> np.ndarray:
        return self.__anomaly_score


print("✅ DLinear EasyTSAD method ready")

✅ DLinear EasyTSAD method ready


In [4]:
# Cell 4 — Create config file for DLinear baseline
import os
config_text = """\
[Data_Params]
preprocess = "z-score"
diff_order = 0


[Model_Params.Default]
window = 96
batch_size = 128
epochs = 50
lr = 0.001
patience = 5
"""


CFG_PATH = os.path.join("kanad", "config_dlinear.toml")
with open(CFG_PATH, "w") as f:
  f.write(config_text)

print("✅ Config written to:", CFG_PATH)
print(open(CFG_PATH).read())

✅ Config written to: kanad/config_dlinear.toml
[Data_Params]
preprocess = "z-score"
diff_order = 0


[Model_Params.Default]
window = 96
batch_size = 128
epochs = 50
lr = 0.001
patience = 5



In [5]:
# Cell 4.5 — Define MTSWindowDataset (REQUIRED for DLinear)

from torch.utils.data import Dataset
import numpy as np
import torch

class MTSWindowDataset(Dataset):
    def __init__(self, tsData, phase, window_size):
        self.window_size = int(window_size)

        if phase == "train":
            self.data = np.asarray(tsData.train)
        elif phase == "valid":
            self.data = np.asarray(tsData.valid)
        elif phase == "test":
            self.data = np.asarray(tsData.test)
        else:
            raise ValueError("phase must be train / valid / test")

        assert self.data.ndim == 2, f"Expected 2D MTS array, got {self.data.shape}"

        self.N, self.F = self.data.shape
        self.sample_num = max(self.N - self.window_size, 0)

    def __len__(self):
        return self.sample_num

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size]
        y = self.data[idx + self.window_size]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
        )

print("✅ MTSWindowDataset ready for DLinear")

✅ MTSWindowDataset ready for DLinear


In [6]:
# Cell 5 — Train DLinear on ORIGINAL SMAP

from EasyTSAD.Controller import TSADController

gctrl = TSADController()

# -----------------------------
# Dataset (ORIGINAL SMAP)
# -----------------------------
gctrl.set_dataset(
    dataset_type="MTS",
    dirname="datasets",
    datasets=["SMAP"],   # original SMAP dataset (no holdout)
)

# -----------------------------
# Method
# -----------------------------
METHOD_NAME = "DLinear_TSAD"
TRAINING_SCHEMA = "naive"

print("🚀 Training DLinear baseline on SMAP...")

gctrl.run_exps(
    method=METHOD_NAME,
    training_schema=TRAINING_SCHEMA,
    cfg_path=CFG_PATH,
)

print("✅ Training finished")

(2026-06-05 22:27:09,218) [INFO]: 
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚══════╝   ╚═╝          ╚═╝   ╚══════╝╚═╝  ╚═╝╚═════╝ 
                                                                      
                         
INFO:logger:
                         
███████╗ █████╗ ███████╗██╗   ██╗    ████████╗███████╗ █████╗ ██████╗ 
██╔════╝██╔══██╗██╔════╝╚██╗ ██╔╝    ╚══██╔══╝██╔════╝██╔══██╗██╔══██╗
█████╗  ███████║███████╗ ╚████╔╝        ██║   ███████╗███████║██║  ██║
██╔══╝  ██╔══██║╚════██║  ╚██╔╝         ██║   ╚════██║██╔══██║██║  ██║
███████╗██║  ██║███████║   ██║          ██║   ███████║██║  ██║██████╔╝
╚══════╝╚═╝  ╚═╝╚═════

🚀 Training DLinear baseline on SMAP...


(2026-06-05 22:27:09,805) [INFO]:     [DLinear_TSAD] handling dataset SMAP | curve AllInOne 
INFO:logger:    [DLinear_TSAD] handling dataset SMAP | curve AllInOne 
Valid 1: 100%|██████████| 1056/1056 [00:03<00:00, 341.39it/s]


Epoch 1 | train=11776.287733 | valid=11456.651479


Valid 2: 100%|██████████| 1056/1056 [00:03<00:00, 309.40it/s]


...

Epoch 7 | train=11453.019067 | valid=11455.229614


Valid 8: 100%|██████████| 1056/1056 [00:03<00:00, 336.56it/s]


Epoch 8 | train=11448.439589 | valid=11449.261422


Valid 9: 100%|██████████| 1056/1056 [00:03<00:00, 347.12it/s]


Epoch 9 | train=11455.354814 | valid=11455.441451
Early stopping
✅ Training finished


In [7]:
# Cell 6 — Evaluate DLinear (same protocol as KAN)

from EasyTSAD.Evaluations.Protocols import (
    EventF1PA,
    PointF1PA,
    PointKthF1PA,
    PointAuprcPA,
)

method = "DLinear_TSAD"
training_schema = "naive"

gctrl.set_evals([
    PointF1PA(),                  # main F1 (paper metric)
    EventF1PA(mode="squeeze"),
    PointKthF1PA(k=5),
    PointAuprcPA(),
])

gctrl.do_evals(method=method, training_schema=training_schema)

print("✅ Evaluation completed")

(2026-06-05 22:28:41,521) [INFO]: Register evaluations
INFO:logger:Register evaluations
(2026-06-05 22:28:41,523) [INFO]: Perform evaluations. Method[DLinear_TSAD], Schema[naive].
INFO:logger:Perform evaluations. Method[DLinear_TSAD], Schema[naive].
(2026-06-05 22:28:41,526) [INFO]:     [Load Data (All)] DataSets: SMAP 
INFO:logger:    [Load Data (All)] DataSets: SMAP 
(2026-06-05 22:28:41,568) [INFO]:     [DLinear_TSAD] Eval dataset SMAP <<<
INFO:logger:    [DLinear_TSAD] Eval dataset SMAP <<<
(2026-06-05 22:28:41,571) [INFO]:         [SMAP] Using default margins (0, 5)
INFO:logger:        [SMAP] Using default margins (0, 5)


✅ Evaluation completed


In [8]:
# Cell 7 — Load DLinear results and convert to comparison row

import json
import os

# -----------------------------
# Paths (IMPORTANT)
# -----------------------------
avg_path = "/content/KAN-AD/Results/Evals/DLinear_TSAD/naive/SMAP/avg.json"
all_path = "/content/KAN-AD/Results/Evals/DLinear_TSAD/naive/SMAP/all.json"

assert os.path.exists(avg_path), f"avg.json not found: {avg_path}"
assert os.path.exists(all_path), f"all.json not found: {all_path}"

# -----------------------------
# Load
# -----------------------------
with open(avg_path, "r") as f:
    avg = json.load(f)

with open(all_path, "r") as f:
    all_scores = json.load(f)

# -----------------------------
# Print (same style as KAN)
# -----------------------------
print("=== 📊 AVERAGE RESULTS (DLinear - SMAP) ===")
for k, v in avg.items():
    print(f"{k}: {v}")

print("\n=== 📊 PER-SERIES RESULTS ===")
print("Number of series:", len(all_scores))
print("Example entry:")
print(list(all_scores.items())[0])

# -----------------------------
# Extract key metrics (for thesis)
# -----------------------------
row = {
    "Model": "DLinear",
    "Family": "Linear",
    "Attention": "No",
    "Dataset": "SMAP",

    # Main metrics
    "F1": avg["best f1 under pa"]["f1"],
    "Precision": avg["best f1 under pa"]["precision"],
    "Recall": avg["best f1 under pa"]["recall"],

    "Event_F1": avg["event-based f1 under pa with mode squeeze"]["f1"],
    "Delay_F1": avg["best f1 under 5-delay pa"]["f1"],

    "AUPRC": avg["point-based auprc pa"],
}

print("\n=== ✅ Clean row (for comparison notebook) ===")
for k, v in row.items():
    print(f"{k}: {v}")

=== 📊 AVERAGE RESULTS (DLinear - SMAP) ===
best f1 under pa: {'f1': 0.7144813732651564, 'precision': 0.7180348327185303, 'recall': 0.7109629118133416, 'threshold': 5.817677617073059}
event-based f1 under pa with mode squeeze: {'f1': 0.3571428571428568, 'precision': 0.8823529411764706, 'recall': 0.22388059701492538, 'threshold': 259.98447716236115}
best f1 under 5-delay pa: {'f1': 0.23411547249249068, 'precision': 0.13267782883330706, 'recall': 0.9942941251294725, 'threshold': 0.0025967955589294434}
point-based auprc pa: 0.8077175254301023

=== 📊 PER-SERIES RESULTS ===
Number of series: 1
Example entry:
('AllInOne', {'best f1 under pa': {'f1': 0.7144813732651564, 'precision': 0.7180348327185303, 'recall': 0.7109629118133416, 'threshold': 5.817677617073059}, 'event-based f1 under pa with mode squeeze': {'f1': 0.3571428571428568, 'precision': 0.8823529411764706, 'recall': 0.22388059701492538, 'threshold': 259.98447716236115}, 'best f1 under 5-delay pa': {'f1': 0.23411547249249068, 'precis